# Baseline 2 — BVP + CNN-5 (Supervised)

**Goal:** Establish a supervised deep learning baseline using a 5-layer 3-D CNN on the
same LORO folds as the SVM baseline.

**Protocol:** 3-fold Leave-One-Room-Out (LORO), standard-6 gestures only.  
**Input:** BVP cube `(1, 20, 20, 20)` — channel-first, already z-score normalised.  
**Architecture:** 5 × Conv3d + BatchNorm + ReLU → GlobalAvgPool → Linear(256 → 6).  

**Two runs per fold:**

| Run | Train set | Purpose |
|---|---|---|
| A | 25% labeled | Our semi-supervised setting |
| B | 100% labeled | Fully supervised upper bound |

**Runtime:** ~30–50 min total on Colab **T4 GPU**.
Enable GPU: Runtime → Change runtime type → T4 GPU.

## 1 — Colab Setup

In [ ]:
# Mount Google Drive (preprocessed.npz lives here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/pi-ssl'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Fatima112299/pi-ssl.git {REPO_DIR}

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2 — Configuration

**Only change `NPZ_PATH`** to wherever you uploaded `preprocessed.npz` on your Drive.

In [ ]:
# ── Path to preprocessed.npz on Google Drive ──────────────────────────────────
NPZ_PATH = '/content/drive/MyDrive/pi-ssl/preprocessed.npz'

# ── Training hyperparameters ──────────────────────────────────────────────────
EPOCHS        = 100
BATCH_SIZE    = 64
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2      # DataLoader workers; 0 if multiprocessing causes errors

RANDOM_SEED   = 42
LABELED_RATIO = 0.25
NUM_CLASSES   = 6      # standard-6 gestures
# ─────────────────────────────────────────────────────────────────────────────

print(f'NPZ_PATH     : {NPZ_PATH}')
print(f'Epochs       : {EPOCHS}')
print(f'Batch size   : {BATCH_SIZE}')
print(f'LR           : {LR}  weight_decay={WEIGHT_DECAY}')
print(f'Labeled ratio: {LABELED_RATIO}')

## 3 — Imports

In [ ]:
import sys, os

REPO_DIR = '/content/pi-ssl'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if not os.path.exists(REPO_DIR):
    raise RuntimeError('Repo not found — run the clone cell first, then re-run this cell.')

import time
import json
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report

from src.data.bvp_dataset import load_npz, BVPDataset
from src.data.splits import make_loeo_splits
from src.data.widar3_dataset import ID_TO_GESTURE

# ── Reproducibility ───────────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU found. Training will be slow. Enable T4 GPU in Runtime settings.')

print('All imports OK.')

## 4 — Load Data

In [ ]:
print('Loading preprocessed.npz ...')
t0 = time.time()
npz_data, file_list = load_npz(NPZ_PATH)
print(f'  Samples   : {len(file_list)}')
print(f'  BVP shape : {npz_data["bvp"].shape}  {npz_data["bvp"].dtype}')
print(f'  Load time : {time.time() - t0:.1f}s')

## 5 — CNN-5 Architecture

```
Input  (B, 1, 20, 20, 20)
  Conv3d(1→32,  k=3, p=1) → BN → ReLU                → (B, 32,  20, 20, 20)
  Conv3d(32→64, k=3, p=1) → BN → ReLU → MaxPool3d(2) → (B, 64,  10, 10, 10)
  Conv3d(64→128,k=3, p=1) → BN → ReLU                → (B, 128, 10, 10, 10)
  Conv3d(128→256,k=3,p=1) → BN → ReLU → MaxPool3d(2) → (B, 256,  5,  5,  5)
  Conv3d(256→256,k=3,p=1) → BN → ReLU                → (B, 256,  5,  5,  5)
  GlobalAvgPool3d                                      → (B, 256)
  Dropout(0.5)
  Linear(256 → 6)                                      → (B, 6)
```

In [ ]:
class CNN5(nn.Module):
    """
    5-layer 3-D convolutional network for BVP gesture classification.

    Input shape  : (batch, 1, 20, 20, 20)  — channel-first BVP cube
    Output shape : (batch, num_classes)    — raw logits

    Architecture follows the CNN-5 design used as a supervised baseline
    in Wi-Fi sensing benchmarks (adapted for 3-D BVP input).
    """

    def __init__(self, num_classes=6, dropout=0.5):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1 — no pooling; preserve resolution
            nn.Conv3d(1,   32,  kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),

            # Block 2 — halve spatial dims: 20→10
            nn.Conv3d(32,  64,  kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),

            # Block 3 — no pooling
            nn.Conv3d(64,  128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),

            # Block 4 — halve spatial dims: 10→5
            nn.Conv3d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),

            # Block 5 — no pooling
            nn.Conv3d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
        )

        # Global average pooling: (B, 256, 5, 5, 5) → (B, 256)
        self.gap = nn.AdaptiveAvgPool3d(1)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)     # (B, 256, 5, 5, 5)
        x = self.gap(x)          # (B, 256, 1, 1, 1)
        x = x.flatten(1)         # (B, 256)
        x = self.classifier(x)   # (B, num_classes)
        return x


# Quick shape check
set_seed(RANDOM_SEED)
_model = CNN5(num_classes=NUM_CLASSES).to(DEVICE)
_dummy = torch.zeros(2, 1, 20, 20, 20).to(DEVICE)
_out   = _model(_dummy)
print(f'CNN-5 output shape : {_out.shape}  (expect [2, {NUM_CLASSES}])')

n_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)
print(f'Trainable params   : {n_params:,}')
del _model, _dummy, _out

## 6 — Training & Evaluation Utilities

In [ ]:
def remap_gesture_ids(file_subset):
    """
    Standard-6 gestures have global IDs 0,1,2,3,4,5 — already contiguous.
    This function verifies that and returns the subset unchanged.
    Raises if any sample has an ID outside 0-5 (which would break CrossEntropyLoss).
    """
    ids = set(f['gesture_id'] for f in file_subset)
    if not ids.issubset(set(range(NUM_CLASSES))):
        raise ValueError(f'Unexpected gesture IDs in split: {ids}')
    return file_subset


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for bvp, labels in loader:
        bvp    = bvp.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(bvp)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss    += loss.item() * len(labels)
        total_correct += (preds == labels).sum().item()
        total_samples += len(labels)

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for bvp, labels in loader:
        bvp    = bvp.to(device, non_blocking=True)
        logits = model(bvp)
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)


def run_cnn_setting(tag, train_files, test_files, npz_data, device,
                    epochs, batch_size, lr, weight_decay, seed, num_classes):
    """
    Train CNN-5 on train_files and evaluate on test_files.
    Returns dict with accuracy, predictions, and training history.
    """
    set_seed(seed)
    print(f'\n  [{tag}]  train={len(train_files)}  test={len(test_files)}')

    # Datasets
    train_ds = BVPDataset(remap_gesture_ids(train_files), npz_data)
    test_ds  = BVPDataset(remap_gesture_ids(test_files),  npz_data)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=NUM_WORKERS,
                              pin_memory=(device.type == 'cuda'), drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size * 2,
                              shuffle=False, num_workers=NUM_WORKERS,
                              pin_memory=(device.type == 'cuda'))

    # Model, optimizer, scheduler, loss
    model     = CNN5(num_classes=num_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    # Training loop
    history = []
    t_start = time.time()

    for epoch in range(1, epochs + 1):
        loss, train_acc = train_one_epoch(model, train_loader, optimizer,
                                          criterion, device)
        scheduler.step()

        if epoch % 10 == 0 or epoch == 1:
            elapsed = time.time() - t_start
            print(f'    Epoch {epoch:>3}/{epochs}  '
                  f'loss={loss:.4f}  train_acc={100*train_acc:.1f}%  '
                  f'elapsed={elapsed:.0f}s')

        history.append({'epoch': epoch, 'loss': loss, 'train_acc': train_acc})

    train_time = time.time() - t_start

    # Evaluation
    y_pred, y_test = evaluate(model, test_loader, device)
    acc = accuracy_score(y_test, y_pred) * 100
    print(f'    Train time : {train_time:.1f}s')
    print(f'    Test acc   : {acc:.2f}%')

    # Per-class report
    labels_present = sorted(set(y_test.tolist()))
    class_names    = [ID_TO_GESTURE[i] for i in labels_present]
    print(f'    Per-class breakdown:')
    report = classification_report(
        y_test, y_pred,
        labels=labels_present, target_names=class_names,
        digits=3, zero_division=0
    )
    for line in report.strip().split('\n'):
        print(f'      {line}')

    return {
        'accuracy'   : acc,
        'train_time' : train_time,
        'y_pred'     : y_pred,
        'y_test'     : y_test,
        'history'    : history,
    }


print('Utilities defined.')

## 7 — Run 3-Fold Evaluation

Each fold trains two CNN-5 models: one with 25% labels, one with 100% labels.

> **Time estimate:** ~5–8 min per model on T4 GPU. All 6 models ≈ 30–50 min total.

In [ ]:
FOLD_ROOMS = {0: 'Office', 1: 'Classroom', 2: 'Hall'}
fold_results = {}
t_total = time.time()

for fold in range(3):
    print(f'\n{"="*62}')
    print(f'FOLD {fold}  —  test room: {FOLD_ROOMS[fold]}')
    print(f'{"="*62}')

    _, labeled, unlabeled, test = make_loeo_splits(
        bvp_root=None, fold=fold,
        labeled_ratio=LABELED_RATIO, seed=RANDOM_SEED,
        file_list=file_list
    )
    full_train = labeled + unlabeled

    print(f'  Labeled train (25%) : {len(labeled)}')
    print(f'  Full train   (100%) : {len(full_train)}')
    print(f'  Test                : {len(test)}')

    fold_results[fold] = {}

    # 25% labeled
    fold_results[fold]['25%_labeled'] = run_cnn_setting(
        tag='25%_labeled', train_files=labeled, test_files=test,
        npz_data=npz_data, device=DEVICE,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        lr=LR, weight_decay=WEIGHT_DECAY,
        seed=RANDOM_SEED, num_classes=NUM_CLASSES
    )

    # 100% labeled
    fold_results[fold]['100%_labeled'] = run_cnn_setting(
        tag='100%_labeled', train_files=full_train, test_files=test,
        npz_data=npz_data, device=DEVICE,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        lr=LR, weight_decay=WEIGHT_DECAY,
        seed=RANDOM_SEED, num_classes=NUM_CLASSES
    )

print(f'\nTotal wall time: {(time.time()-t_total)/60:.1f} min')

## 8 — Summary Table

In [ ]:
accs_25  = [fold_results[f]['25%_labeled']['accuracy']  for f in range(3)]
accs_100 = [fold_results[f]['100%_labeled']['accuracy'] for f in range(3)]

mean_25  = np.mean(accs_25)
std_25   = np.std(accs_25)
mean_100 = np.mean(accs_100)
std_100  = np.std(accs_100)

print('\n' + '='*65)
print(f'{"RESULTS SUMMARY — CNN-5":^65}')
print('='*65)
print(f'{"Fold":<6} {"Test Room":<12} {"CNN-5 25% labels":>18} {"CNN-5 100% labels":>19}')
print('-'*65)
for f in range(3):
    a25  = fold_results[f]['25%_labeled']['accuracy']
    a100 = fold_results[f]['100%_labeled']['accuracy']
    print(f'{f:<6} {FOLD_ROOMS[f]:<12} {a25:>17.2f}%  {a100:>17.2f}%')
print('-'*65)
print(f'{"Mean":<18} {mean_25:>17.2f}%  {mean_100:>17.2f}%')
print(f'{"Std":<18} {std_25:>17.2f}%  {std_100:>17.2f}%')
print('='*65)
print()
print('Comparison with SVM baseline:')
print(f'  SVM  25% labels : 62.59% ± 8.85%')
print(f'  SVM 100% labels : 69.03% ± 7.76%')
print(f'  CNN-5 25% labels  (this run) : {mean_25:.2f}% ± {std_25:.2f}%')
print(f'  CNN-5 100% labels (this run) : {mean_100:.2f}% ± {std_100:.2f}%')
print()
print('PI-SSL target (25% labels) : >89.50%')

## 9 — Per-Gesture Accuracy Breakdown

In [ ]:
from collections import defaultdict

for setting in ['25%_labeled', '100%_labeled']:
    gesture_correct = defaultdict(int)
    gesture_total   = defaultdict(int)

    for f in range(3):
        y_test = fold_results[f][setting]['y_test']
        y_pred = fold_results[f][setting]['y_pred']
        for true, pred in zip(y_test, y_pred):
            gesture_total[true]   += 1
            gesture_correct[true] += int(true == pred)

    print(f'Per-gesture accuracy ({setting}, averaged across 3 folds):')
    print(f'  {"Gesture":>22}   {"Correct":>8}   {"Total":>7}   {"Accuracy":>9}')
    print(f'  {"-"*56}')
    for gid in sorted(gesture_total):
        name = ID_TO_GESTURE[gid]
        acc  = 100 * gesture_correct[gid] / gesture_total[gid]
        print(f'  {name:>22}   {gesture_correct[gid]:>8}   '
              f'{gesture_total[gid]:>7}   {acc:>8.2f}%')
    print()

## 10 — Save Results

In [ ]:
RESULTS_PATH = '/content/drive/MyDrive/pi-ssl/results_cnn5_baseline.json'

summary = {
    'method'        : 'BVP+CNN5',
    'epochs'        : EPOCHS,
    'batch_size'    : BATCH_SIZE,
    'lr'            : LR,
    'weight_decay'  : WEIGHT_DECAY,
    'labeled_ratio' : LABELED_RATIO,
    'seed'          : RANDOM_SEED,
    'folds': {
        str(f): {
            'test_room'        : FOLD_ROOMS[f],
            'acc_25pct'        : fold_results[f]['25%_labeled']['accuracy'],
            'acc_100pct'       : fold_results[f]['100%_labeled']['accuracy'],
            'train_time_25pct' : fold_results[f]['25%_labeled']['train_time'],
            'train_time_100pct': fold_results[f]['100%_labeled']['train_time'],
        }
        for f in range(3)
    },
    'mean_acc_25pct'  : float(mean_25),
    'std_acc_25pct'   : float(std_25),
    'mean_acc_100pct' : float(mean_100),
    'std_acc_100pct'  : float(std_100),
}

with open(RESULTS_PATH, 'w') as fp:
    json.dump(summary, fp, indent=2)

print(f'Results saved to {RESULTS_PATH}')
print(json.dumps(summary, indent=2))

---
## Interpretation

| Method | Labels | Mean Accuracy |
|---|---|---|
| SVM (raw BVP) | 25% | 62.59% ± 8.85% |
| SVM (raw BVP) | 100% | 69.03% ± 7.76% |
| **CNN-5** | **25%** | **(your result)** |
| **CNN-5** | **100%** | **(your result)** |
| **PI-SSL target** | **25%** | **>89.50%** |

**What to look for:**
- CNN-5 should outperform SVM at both 25% and 100% labels — if not, check architecture or training.
- The gap between CNN-5 25% and CNN-5 100% is the **label efficiency problem** PI-SSL must solve.
- PI-SSL's goal: with only 25% labels, match or exceed CNN-5 at 100% labels.

**Next step:** Implement PI-SSL model in `src/models/`.